In [13]:
# Célula 1: Instalação de Dependências
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install beautifulsoup4 lxml pandas openpyxl requests

In [14]:
# Célula 2: Importações do Sistema e Bibliotecas de Terceiros
import os
import re
import time
import warnings
import xml.etree.ElementTree as ET
from typing import List, Dict, Any, Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup

# Configurações do Pandas para exibição no Notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
warnings.filterwarnings('ignore')

print("Transformação de dados farmacológicos inicializada com sucesso.")

Transformação de dados farmacológicos inicializada com sucesso.


In [15]:
# Célula 3: Componente de Conexão com o PubMed e Conversor PMC
class PubMedClient:
    """Gerencia a comunicação robusta com a API Entrez e ID Converter do NCBI."""
    BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    CONVERTER_URL = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"
    
    @staticmethod
    def buscar_ids(termo: str, max_resultados: int = 5000) -> List[str]:
        """Executa o esearch para coletar IDs de artigos dentro do escopo."""
        url = f"{PubMedClient.BASE_URL}esearch.fcgi"
        parametros = {
            "db": "pubmed",
            "term": termo,
            "retmax": max_resultados,
            "retmode": "json"
        }
        try:
            resposta = requests.get(url, params=parametros, timeout=30)
            resposta.raise_for_status()
            dados = resposta.json()
            ids = dados.get("esearchresult", {}).get("idlist", [])
            print(f"[Sucesso] Total de IDs localizados na busca: {len(ids)}")
            return ids
        except Exception as e:
            print(f"[Erro] Falha ao consultar esearch: {e}")
            return []

    @staticmethod
    def baixar_lote_xml(ids_lote: List[str]) -> Optional[str]:
        """Executa o efetch para extrair o XML em lotes e evitar sobrecarga."""
        url = f"{PubMedClient.BASE_URL}efetch.fcgi"
        parametros = {
            "db": "pubmed",
            "id": ",".join(ids_lote),
            "retmode": "xml",
            "rettype": "abstract"
        }
        try:
            resposta = requests.post(url, data=parametros, timeout=45)
            resposta.raise_for_status()
            return resposta.text
        except Exception as e:
            print(f"[Erro] Falha no efetch para o lote atual: {e}")
            return None

    @staticmethod
    def converter_pmid_para_pmc(pmids: List[str]) -> Dict[str, str]:
        """Converte uma lista de PMIDs para o formato estrito PMCxxxx."""
        if not pmids:
            return {}
        parametros = {
            "ids": ",".join(pmids),
            "format": "json",
            "tool": "pipeline_mineracao",
            "email": "mineracao@farmaco.com"
        }
        try:
            resposta = requests.get(PubMedClient.CONVERTER_URL, params=parametros, timeout=20)
            resposta.raise_for_status()
            dados = resposta.json()
            mapa_conversao = {}
            for registro in dados.get("records", []):
                pmid = registro.get("pmid")
                pmcid = registro.get("pmcid")
                if pmid and pmcid:
                    mapa_conversao[str(pmid)] = str(pmcid)
            return mapa_conversao
        except Exception:
            return {} # Retorna dicionário vazio em caso de erro ou ausência de indexação no PMC

In [16]:
# Célula 4: Motores de Higienização Textual e Normalização
def normalizar_texto(texto: str) -> str:
    """Normaliza o texto para minúsculas e remove espaçamentos redundantes."""
    if not texto:
        return ""
    texto = texto.lower()
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

def remover_stopwords(texto_substancia: str, stopwords: List[str]) -> str:
    """Elimina ruídos e termos botânicos/químicos genéricos do nome do composto."""
    padrao = r'\b(' + '|'.join(map(re.escape, stopwords)) + r')\b'
    resultado = re.sub(padrao, '', texto_substancia)
    return re.sub(r'\s+', ' ', resultado).strip()

In [17]:
# Célula 5: Classe Especialista de Extração com Filtro Sintático Avançado (V4)
class FarmacoExtrator:
    """Aplica regras avançadas de NLP e Farmacologia para purificação do Dataset."""
    
    def __init__(self, constantes: dict):
        self.c = constantes
        # Lista de conectivos e ruídos sintáticos que frequentemente antecedem uma dose
        self.ruidos_sintaticos = {
            'with', 'that', 'individually', 'doses', 'dose', 'at', 'of', 'and', 'group', 
            'groups', 'acid', 'administered', 'received', 'treated', 'injected', 'mg', 'kg'
        }

    def extrair_dose_e_composto(self, abstract_txt: str, titulo_txt: str) -> tuple:
        """Localiza a dose e isola quimicamente a substância mitigando falsos positivos."""
        match_dose = re.search(self.c['REGEX_DOSE'], abstract_txt)
        if not match_dose:
            return None, None
        
        dose_val = float(match_dose.group(1))
        posicao_dose = match_dose.start()
        
        # Janela retrospectiva ajustada para capturar até 3 palavras anteriores (Bigrams/Trigrams)
        inicio_janela = max(0, posicao_dose - 100)
        janela_contexto = abstract_txt[inicio_janela:posicao_dose]
        
        # Filtro de segurança contra a Lista Negra Farmacológica expandida
        for item_bloqueado in self.c['LISTA_NEGRA']:
            if item_bloqueado in janela_contexto:
                return None, None
        
        # Tokenização focada em capturar estruturas de nomes de compostos (incluindo hifens e números, ex: MK-801)
        tokens = re.findall(r'\b[a-zA-Z0-9\-]{3,25}\b', janela_contexto)
        if not tokens:
            return None, dose_val
            
        # Remove stopwords e pontuações do final da lista de tokens
        tokens_filtrados = [t for t in tokens if t not in self.c['STOPWORDS']]
        if not tokens_filtrados:
            return None, dose_val
            
        # Tenta avaliar o último e o penúltimo token para evitar preposições como "with"
        candidato = tokens_filtrados[-1].strip()
        
        # Se o candidato for um ruído sintático isolado, tenta pegar a palavra anterior a ele
        if candidato in self.ruidos_sintaticos and len(tokens_filtrados) > 1:
            candidato = tokens_filtrados[-2].strip()
            
        # Se ainda assim cair em um ruído sintático, o dado é descartado como "Lixo"
        if candidato in self.ruidos_sintaticos or candidato.isdigit():
            return None, dose_val
            
        # Validação Cruzada de Confiança: O Composto precisa ter relevância no Artigo (Título ou Abstract)
        if candidato in titulo_txt or candidato in abstract_txt[:400]:
            return candidato, dose_val
            
        return None, dose_val

    def inferir_via_administracao(self, abstract_txt: str) -> Optional[str]:
        """Detecta a via de administração ou padroniza como p.o. (Default comum)."""
        for via, padrao in self.c['REGEX_VIAS'].items():
            if re.search(padrao, abstract_txt):
                return via
        return "not_specified" # Remove células vazias na via de administração

    def avaliar_fenotipos(self, abstract_txt: str) -> dict:
        """Mapeia os fenótipos forçando densidade de matriz (0 em vez de None) para ML."""
        tem_estatistica = any(re.search(p, abstract_txt) for p in self.c['ESTATISTICA'])
        tem_falha = any(re.search(f, abstract_txt) for f in self.c['FALHA'])
        
        # Para evitar células vazias, inicializamos o Baseline como 0 (Ausência de efeito detectado)
        fenotipos = {
            'aumento_latencia_crise': 0,
            'reducao_severidade_racine': 0,
            'protecao_contra_morte': 0,
            'target_anticonvulsivante': 0
        }
        
        # Se houver indicação de falha explícita, mantemos tudo em 0 e encerramos
        if tem_falha:
            return fenotipos
            
        # Casos positivos validados estatisticamente
        if tem_estatistica:
            if any(re.search(p, abstract_txt) for p in self.c['P_LATENCIA']):
                fenotipos['aumento_latencia_crise'] = 1
                
            if any(re.search(p, abstract_txt) for p in self.c['P_SEVERIDADE']):
                fenotipos['reducao_severidade_racine'] = 1
                
            if any(re.search(p, abstract_txt) for p in self.c['P_MORTE']):
                fenotipos['protecao_contra_morte'] = 1

        # Consolidação lógica do Target da IA
        if 1 in [fenotipos['aumento_latencia_crise'], fenotipos['reducao_severidade_racine'], fenotipos['protecao_contra_morte']]:
            fenotipos['target_anticonvulsivante'] = 1
            
        return fenotipos

In [18]:
# Célula 6: Configuração de Constantes Globais de Negócio e Engenharia (V3 - Proteção PTZ)
MODO_TESTE = False  # Altere para False para rodar toda a base histórica mundial

CONSTANTES = {
    'TERMO_BUSCA': (
        '(pentylenetetrazole[Title/Abstract] OR PTZ[Title/Abstract]) AND '
        '(seizures[Title/Abstract] OR convulsion[Title/Abstract] OR convulsions[Title/Abstract]) AND '
        '(mice[Title/Abstract] OR mouse[Title/Abstract]) AND '
        '(anticonvulsant[Title/Abstract] OR anticonvulsants[Title/Abstract] OR neuroprotection[Title/Abstract] '
        'OR neuroprotective[Title/Abstract] OR antiepileptic[Title/Abstract] OR antiepileptics[Title/Abstract]) '
        'AND ("2011/01/01"[Date - Publication] : "2026/12/31"[Date - Publication]) NOT Review[Publication Type]'
    ),
    
    # LISTA NEGRA FARMACOLÓGICA: Impede terminantemente a extração do indutor ou de fármacos controle
    'LISTA_NEGRA': [
        'pentylenetetrazole', 'ptz', 'pentylenetetrazol',                      # Agentes indutores (Gatilhos)
        'diazepam', 'valproate', 'phenytoin', 'phenobarbital', 'carbamazepine', # Controles positivos (Padrões)
        'clonazepam', 'ethosuximide', 'lorazepam', 'midazolam',                # Outros anticonvulsivantes padrão
        'saline', 'vehicle', 'tween', 'water', 'control'                       # Controles negativos / Veículos
    ],
    
    'STOPWORDS': ['extract', 'fraction', 'essential oil', 'treatment', 'group', 'isolated'],
    
    # Regex e Padrões Linguísticos
    'REGEX_DOSE': r'(\d+(?:\.\d+)?)\s*mg/kg',
    'REGEX_VIAS': {
        'i.p.': r'\bi\.p\.\b|\bintraperitoneal\b',
        'p.o.': r'\bp\.o\.\b|\boral\b|\borally\b'
    },
    'ESTATISTICA': [r'p\s*<\s*0\.05', r'significantly', r'attenuated', r'increased', r'prevented'],
    'FALHA': [r'did not alter', r'no significant effect', r'no effect', r'failed to protect'],
    
    # Heurísticas de Validação Fenotípica
    'P_LATENCIA': [r'latency.*(increased|delayed|prolonged)', r'(prolongation|increase).*latency'],
    'P_SEVERIDADE': [r'(intensity|severity|score).*(reduced|attenuated|decreased)', r'(reduction|attenuation).*severity'],
    'P_MORTE': [r'(death|mortality|lethality).*(protected|prevented|reduced|survival)', r'protection against.*(death|mortality)']
}

print(f"Configuração e Lista Negra farmacológica atualizadas. Modo Teste Ativo? {MODO_TESTE}")

Configuração e Lista Negra farmacológica atualizadas. Modo Teste Ativo? False


In [25]:
# Célula 7: Extrator do Escopo e Parseador XML com Formatação Compacta de ID
def processar_xml_pubmed(xml_data: str, extrator: FarmacoExtrator) -> List[Dict[str, Any]]:
    """Faz o parse do XML com fallback seguro e formatação compacta de ID (Ex: PMID123456)."""
    registros = []
    if not xml_data:
        return registros
        
    soup = BeautifulSoup(xml_data, "xml")
    artigos = soup.find_all("PubmedArticle")
    
    lista_pmids = [artigo.find("PMID").text.strip() for artigo in artigos if artigo.find("PMID")]
    mapa_pmc = PubMedClient.converter_pmid_para_pmc(lista_pmids)
    
    for artigo in artigos:
        pmid_el = artigo.find("PMID")
        if not pmid_el:
            continue
        pmid = pmid_el.text.strip()
        
        # Consolidação do ID sem o caractere "_" conforme especificado (Ex: PMID42192296)
        pmc_id = mapa_pmc.get(pmid)
        id_referencia = pmc_id if pmc_id else f"PMID{pmid}"
            
        titulo_el = artigo.find("ArticleTitle")
        titulo_limpo = normalizar_texto(titulo_el.text if titulo_el else "")
        
        tags_abstract = artigo.find_all("AbstractText")
        abstract_completo = " ".join([tag.text for tag in tags_abstract if tag.text])
        abstract_limpo = normalizar_texto(abstract_completo)
        
        if not abstract_limpo:
            continue
            
        composto, dose = extrator.extrair_dose_e_composto(abstract_limpo, titulo_limpo)
        if not composto and MODO_TESTE is False:
            continue
            
        via = extrator.inferir_via_administracao(abstract_limpo)
        fenotipos = extrator.avaliar_fenotipos(abstract_limpo)
        
        registro = {
            "pmid_referencia": id_referencia, 
            "substancia_testada": composto,
            "dose_mg_kg": dose,
            "via_administration": via,
            "aumento_latencia_crise": fenotipos['aumento_latencia_crise'],
            "reducao_severidade_racine": fenotipos['reducao_severidade_racine'],
            "protecao_contra_morte": fenotipos['protecao_contra_morte'],
            "target_anticonvulsivante": fenotipos['target_anticonvulsivante']
        }
        registros.append(registro)
        
    return registros

In [26]:
# Célula 8: Motor Iterativo em Lotes
def executar_pipeline_extracao(ids_totais: List[str], extrator: FarmacoExtrator, tamanho_lote: int = 50) -> pd.DataFrame:
    """Consome a API e processa os dados respeitando os limites do NCBI."""
    dados_finais = []
    total_ids = len(ids_totais)
    
    print(f"[Início] Iniciando processamento de {total_ids} artigos em lotes de {tamanho_lote}...")
    
    for i in range(0, total_ids, tamanho_lote):
        lote_atual = ids_totais[i:i + tamanho_lote]
        print(f"Processando lote {i // tamanho_lote + 1} ({i} até {min(i + tamanho_lote, total_ids)})...")
        
        xml_resposta = PubMedClient.baixar_lote_xml(lote_atual)
        if xml_resposta:
            registros_lote = processar_xml_pubmed(xml_resposta, extrator)
            dados_finais.extend(registros_lote)
            
        time.sleep(1.0) # Delay antibloqueio mandatório por IP
        
    return pd.DataFrame(dados_finais)

In [27]:
# Célula 9: Coleta de Chaves Primárias Globais
extrator_farmaco = FarmacoExtrator(CONSTANTES)
lista_ids_global = PubMedClient.buscar_ids(CONSTANTES['TERMO_BUSCA'], max_resultados=5000)

[Sucesso] Total de IDs localizados na busca: 705


In [28]:
# Célula 10: Homologação no MODO_TESTE ou Execução Completa
if MODO_TESTE:
    print("[MODO TESTE ATIVO] Executando pipeline apenas para os 50 primeiros IDs coletados...")
    ids_para_processar = lista_ids_global[:50]
else:
    print("[PRODUÇÃO ATIVA] Executando pipeline para toda a literatura mundial mapeada...")
    ids_para_processar = lista_ids_global

df_farmaco = executar_pipeline_extracao(ids_para_processar, extrator_farmaco, tamanho_lote=50)

[PRODUÇÃO ATIVA] Executando pipeline para toda a literatura mundial mapeada...
[Início] Iniciando processamento de 705 artigos em lotes de 50...
Processando lote 1 (0 até 50)...
Processando lote 2 (50 até 100)...
Processando lote 3 (100 até 150)...
Processando lote 4 (150 até 200)...
Processando lote 5 (200 até 250)...
Processando lote 6 (250 até 300)...
Processando lote 7 (300 até 350)...
Processando lote 8 (350 até 400)...
Processando lote 9 (400 até 450)...
Processando lote 10 (450 até 500)...
Processando lote 11 (500 até 550)...
Processando lote 12 (550 até 600)...
Processando lote 13 (600 até 650)...
Processando lote 14 (650 até 700)...
Processando lote 15 (700 até 705)...


In [29]:
# Célula 11: Auditoria de Dados e Estatística do Target
print(f"Quantidade total de registros estruturados: {df_farmaco.shape[0]}")
print("\n--- Distribuição das Classes do Target da IA ---")
if not df_farmaco.empty:
    print(df_farmaco['target_anticonvulsivante'].value_counts(dropna=False))
    print("\n--- Amostra Inicial dos Dados Estruturados ---")
    display(df_farmaco.head(10))
else:
    print("[Alerta] O DataFrame final retornou vazio para os filtros aplicados.")

Quantidade total de registros estruturados: 109

--- Distribuição das Classes do Target da IA ---
target_anticonvulsivante
0    79
1    30
Name: count, dtype: int64

--- Amostra Inicial dos Dados Estruturados ---


,pmid_referencia,substancia_testada,dose_mg_kg,via_administration,aumento_latencia_crise,reducao_severidade_racine,protecao_contra_morte,target_anticonvulsivante
0,PMID42192296,sas,200.0,not_specified,0,1,0,1
1,PMC12477776,emodin,200.0,not_specified,0,1,0,1
2,PMID40437665,stv,100.0,not_specified,0,0,0,0
3,PMC12062037,neuroprotective,5.0,not_specified,0,0,0,0
4,PMID39869286,dapsone,20.0,not_specified,1,0,0,1
5,PMC11191135,brv,20.0,not_specified,0,1,0,1
6,PMC10834591,schb,60.0,not_specified,0,0,0,0
7,PMC10179922,metformin,100.0,not_specified,0,0,0,0
8,PMID36853856,apz,10.0,not_specified,0,0,1,1
9,PMC10029819,kpp-iii-34,500.0,i.p.,0,0,0,0


In [31]:
# Célula 12: Exportação e Fechamento do Pipeline de Dados
arquivo_csv = "dataset_baseline_ptz.csv"
arquivo_xlsx = "dataset_baseline_ptz.xlsx"

if not df_farmaco.empty:
    # Exportação estrita conforme especificado no escopo do projeto
    df_farmaco.to_csv(arquivo_csv, index=False, encoding='utf-8-sig')
    df_farmaco.to_excel(arquivo_xlsx, index=False, engine='openpyxl')
    print(f"[Sucesso Realizado] Arquivos salvos:\n -> {arquivo_csv}\n -> {arquivo_xlsx}")
else:
    print("[Erro no Armazenamento] Arquivos não gravados por ausência de dados válidos coletados.")

[Sucesso Realizado] Arquivos salvos:
 -> dataset_baseline_ptz.csv
 -> dataset_baseline_ptz.xlsx
